# Predicting Price Movements

Modernized for Python 3.10+ without legacy optional dependencies.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
FOLDER = Path.cwd().resolve()
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))
from linear_model_utils import regression_data, classification_data, panel_factor_data, ols_numpy, rank_ic
OUTPUT = FOLDER.parent / 'data' / 'linear_models'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

panel = panel_factor_data(periods=220)
factors = [c for c in panel.columns if c.startswith('factor_')]
panel['direction'] = (panel['return_1d'] > 0).astype(int)
x = panel[factors]
y = panel['direction']
pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
ts = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(pipe, x, y, cv=ts, scoring='accuracy')
summary = pd.DataFrame({'split': range(1, len(scores)+1), 'accuracy': scores})
summary.to_csv(OUTPUT / 'price_direction_logistic_cv.csv', index=False)
display(summary)
display(summary['accuracy'].describe().to_frame())

,split,accuracy
0,1,0.491818
1,2,0.525455
2,3,0.488182
3,4,0.490000
4,5,0.516364


,accuracy
count,5.000000
mean,0.502364
std,0.017280
min,0.488182
25%,0.490000
50%,0.491818
75%,0.516364
max,0.525455
